# Gaussian Processes — Duffing
## Identificação discreta com estados completos

**Projeto:** Data-Driven Dynamics and Control.  
**Origem:** extensão experimental nossa do estudo de GP; não é um experimento do artigo-base.

**Objetivo:** aprender a transição $x_{k+1}=F(x_k,u_k)$ usando dois GPs escalares, sem revelar ao identificador as equações do Duffing. Avaliar predição de um passo, rollout livre, generalização e incerteza.

**Fundamentos e reprodução do Exemplo 4.3:** [GP_Problema_1.ipynb](GP_Problema_1.ipynb).  
**Artigo-base:** [Control-Oriented System Identification: Classical, Learning, and Physics-Informed Approaches](https://arxiv.org/html/2512.06315).

Este notebook usa apenas NumPy, SciPy e Matplotlib. Execute as células de cima para baixo. Não há GPSSM, GP-NARX, redes neurais ou restrições de estabilidade.


## 1. Definindo a planta e o problema

Preservamos o benchmark modificado utilizado nos scripts de Koopman e SINDy do repositório:
$$
\frac{dq}{dt}=p,\qquad
\frac{dp}{dt}=-2p-q\cos(q+p)+u.
$$

O nome Duffing é mantido por continuidade histórica, embora este não seja o oscilador cúbico canônico.

Usamos $dt=0.01$ s e RK4, mantendo $u_k$ constante em cada intervalo. O **simulador** conhece as equações acima, mas o **identificador** observará somente
$$
z_k=[q_k,p_k,u_k]^\top\in\mathbb R^3,\qquad
x_{k+1}=[q_{k+1},p_{k+1}]^\top\in\mathbb R^2.
$$
Nossa escolha é predição direta do próximo estado, **não** do incremento.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize

DT, DELTA = 0.01, 2.0
rng = np.random.default_rng(156)

def dynamics(x, u):
    """Exclusiva do simulador; não é passada ao GP."""
    q, p = x[...,0], x[...,1]
    return np.stack((p, -DELTA*p-q*np.cos(q+p)+u), axis=-1)

def simulate_step(x, u):
    """Passo RK4 com entrada constante; aceita amostras em lote."""
    k1 = dynamics(x,u)
    k2 = dynamics(x+DT*k1/2,u)
    k3 = dynamics(x+DT*k2/2,u)
    k4 = dynamics(x+DT*k3,u)
    return x+DT*(k1+2*k2+2*k3+k4)/6


## 2. Dados experimentais

Um GP exato exige fatoração da matriz $K\in\mathbb R^{N\times N}$, com custo aproximado $O(N^3)$. Por isso partimos de **400 transições independentes**, não das 40 mil usadas em Koopman.

Amostramos $q_k,p_k,u_k$ independentemente e uniformemente em $[-1,1]$, e integramos a planta por um passo para obter $x_{k+1}$.

Nesta primeira rodada não adicionamos ruído de medição. A regularização diagonal do GP, definida adiante, **não representa ruído de medição simulado**. Para uma comparação futura com Koopman, deveremos igualar número de dados e condições de avaliação.


In [ ]:
N_TRAIN = 400  # experiência futura: 150, 250 ou 600
X_train = rng.uniform(-1,1,size=(N_TRAIN,2))
U_train = rng.uniform(-1,1,size=N_TRAIN)
Y_train = simulate_step(X_train,U_train)
Z_train = np.column_stack((X_train,U_train))
assert Z_train.shape==(N_TRAIN,3) and Y_train.shape==(N_TRAIN,2)

fig,axes=plt.subplots(1,2,figsize=(11,4))
sc=axes[0].scatter(X_train[:,0],X_train[:,1],c=U_train,s=15)
fig.colorbar(sc,ax=axes[0],label="u_k")
axes[0].set(xlabel="q_k",ylabel="p_k",title="Cobertura dos dados")
axes[1].scatter(X_train[:,0],Y_train[:,0],s=10,label="q_k → q_{k+1}")
axes[1].scatter(X_train[:,1],Y_train[:,1],s=10,label="p_k → p_{k+1}")
axes[1].set(xlabel="Estado atual",ylabel="Próximo estado",title="Pares de regressão")
axes[1].legend();plt.tight_layout();plt.show()


## 3. Dois GPs escalares com Matérn 3/2

Em vez de um modelo de saída vetorial, treinamos **dois GPs independentes**:
$$
q_{k+1}=f_q(z_k),\qquad p_{k+1}=f_p(z_k),
\quad f_q\sim\mathcal{GP}(m_q,k_q),\quad
f_p\sim\mathcal{GP}(m_p,k_p).
$$

Escolhemos Matérn 3/2 com escalas separadas (ARD):
$$
k(z,z')=\sigma_f^2(1+r)e^{-r},\qquad
r=\sqrt{3\sum_{d=1}^3\left(\frac{z_d-z_d'}{\ell_d}\right)^2}.
$$

As variáveis de entrada e saída serão padronizadas com **médias e desvios-padrão calculados somente no treinamento**. Em coordenadas padronizadas a média a priori é zero; nas unidades físicas ela equivale à média constante do conjunto de treinamento.

A matriz é regularizada por $\sigma_n^2 I$, com $\sigma_n=0.018$ em unidades **normalizadas de saída**. Como a planta gerou dados sem ruído, isso funciona aqui como um pequeno termo de regularização. Estimamos $\ell_q,\ell_p,\ell_u,\sigma_f$ pela verossimilhança marginal.


In [ ]:
Z_MEAN = Z_train.mean(axis=0)
Z_SCALE = Z_train.std(axis=0)
Z_norm = (Z_train-Z_MEAN)/Z_SCALE

def matern32_ard(A,B,ell,sigma_f):
    A=np.atleast_2d(np.asarray(A,float))
    B=np.atleast_2d(np.asarray(B,float))
    D=(A[:,None,:]-B[None,:,:])/ell
    r=np.sqrt(3*np.sum(D*D,axis=-1))
    return sigma_f**2*(1+r)*np.exp(-r)

class GPRegressor:
    """GPR exato escalar: Cholesky + Matérn ARD + máxima verossimilhança."""
    def __init__(self, noise_std=0.018):
        self.noise_std=noise_std

    def fit(self,X,y):
        self.X=np.asarray(X,float)
        y=np.asarray(y,float).reshape(-1)
        self.y_mean=y.mean()
        self.y_std=y.std()
        self.y=(y-self.y_mean)/self.y_std
        n,d=self.X.shape

        def nll(v,factors=False):
            ell=np.exp(v[:d])
            sf=np.exp(v[d])
            K=matern32_ard(self.X,self.X,ell,sf)
            K[np.diag_indices(n)]+=self.noise_std**2+1e-7
            try:
                cf=cho_factor(K,lower=True,check_finite=False)
                alpha=cho_solve(cf,self.y,check_finite=False)
            except np.linalg.LinAlgError:
                return 1e12
            if factors:
                return ell,sf,cf,alpha
            return float(.5*self.y@alpha+np.log(np.diag(cf[0])).sum()
                         +n/2*np.log(2*np.pi))

        initial=np.log([1.0]*d+[1.0])
        result=minimize(nll,initial,method="L-BFGS-B",
                        bounds=[(-2.3,2.3)]*d+[(-2.3,2.3)],
                        options={"maxiter":55})
        self.ell,self.sf,self.cf,self.alpha=nll(result.x,True)
        self.result=result
        return self

    def predict(self,X):
        X=np.atleast_2d(np.asarray(X,float))
        Ks=matern32_ard(X,self.X,self.ell,self.sf)
        mu=Ks@self.alpha
        V=cho_solve(self.cf,Ks.T,check_finite=False)
        var=np.maximum(0,self.sf**2-np.sum(Ks*V.T,axis=1))
        return self.y_mean+self.y_std*mu,self.y_std*np.sqrt(var)

gps=[GPRegressor().fit(Z_norm,Y_train[:,j]) for j in range(2)]
for name,gp in zip(["q_{k+1}","p_{k+1}"],gps):
    print(name,"ell=",np.round(gp.ell,3),"sigma_f=",round(gp.sf,3),
          "otimizador_sucesso=",gp.result.success)

def gp_step(Z):
    """Prediz a média e o desvio-padrão de um próximo estado, em unidades físicas."""
    Zn=(np.atleast_2d(np.asarray(Z,float))-Z_MEAN)/Z_SCALE
    results=[gp.predict(Zn) for gp in gps]
    return (np.column_stack([r[0] for r in results]),
            np.column_stack([r[1] for r in results]))


## 4. One-step: o modelo consegue prever uma transição?

Geramos 300 pontos de teste **não utilizados no treinamento**. Para cada um, o GP recebe o estado verdadeiro $x_k$ e a entrada $u_k$, e estima $x_{k+1}$.

Comparamos ao baseline de **persistência** $\hat x_{k+1}=x_k$. Com $dt$ pequeno, esse baseline já pode apresentar erro baixo; portanto um RMSE baixo sozinho não demonstra boa modelagem.

A banda $\mu\pm1.96\sigma$ é **pontual, condicional ao modelo e aos hiperparâmetros ajustados**. Ela não é uma garantia para a planta.


In [ ]:
rng_test=np.random.default_rng(2026)
N_TEST=300
X_test=rng_test.uniform(-1,1,(N_TEST,2))
U_test=rng_test.uniform(-1,1,N_TEST)
Z_test=np.column_stack((X_test,U_test))
Y_test=simulate_step(X_test,U_test)
P_test,SD_test=gp_step(Z_test)

def rmse(A,B):
    return np.sqrt(np.mean((A-B)**2,axis=0))

print("GP RMSE one-step [q,p]:",rmse(P_test,Y_test))
print("Persistência RMSE one-step [q,p]:",rmse(X_test,Y_test))
print("Cobertura pontual empírica (não garantida):",
      np.mean(np.abs(P_test-Y_test)<=1.96*SD_test,axis=0))

fig,axes=plt.subplots(1,2,figsize=(10,4))
for j,name in enumerate(["q","p"]):
    axes[j].scatter(Y_test[:,j],P_test[:,j],s=12,alpha=.5)
    a=min(Y_test[:,j].min(),P_test[:,j].min())
    b=max(Y_test[:,j].max(),P_test[:,j].max())
    axes[j].plot([a,b],[a,b],"k--")
    axes[j].set(xlabel="Real",ylabel="Predição de um passo",title=name)
    axes[j].grid(alpha=.25)
plt.tight_layout();plt.show()


## 5. Generalização e estimativa de incerteza

Repetimos o teste com $p_k\in[1.1,1.5]$, região **não observada** no treinamento. As demais coordenadas permanecem no intervalo original.

Queremos comparar tanto o erro de predição quanto a incerteza posterior. Uma incerteza pequena **não é prova de que o modelo esteja correto**, sobretudo ao extrapolar.


In [ ]:
X_ood=np.column_stack((rng_test.uniform(-1,1,N_TEST),
                       rng_test.uniform(1.1,1.5,N_TEST)))
U_ood=rng_test.uniform(-1,1,N_TEST)
Z_ood=np.column_stack((X_ood,U_ood))
Y_ood=simulate_step(X_ood,U_ood)
P_ood,SD_ood=gp_step(Z_ood)

print("Dentro:  RMSE",rmse(P_test,Y_test),"σ médio",SD_test.mean(axis=0))
print("Fora:    RMSE",rmse(P_ood,Y_ood),"σ médio",SD_ood.mean(axis=0))

fig,axes=plt.subplots(1,2,figsize=(11,4))
for j,name in enumerate(["q","p"]):
    axes[j].scatter(SD_test[:,j],np.abs(P_test[:,j]-Y_test[:,j]),
                    alpha=.5,s=14,label="dentro")
    axes[j].scatter(SD_ood[:,j],np.abs(P_ood[:,j]-Y_ood[:,j]),
                    alpha=.5,s=14,label="fora")
    axes[j].set(xlabel="Desvio-padrão previsto",ylabel="Erro absoluto",
                title=f"Componente {name}")
    axes[j].legend();axes[j].grid(alpha=.2)
plt.tight_layout();plt.show()


## 6. Rollout livre: uma pergunta distinta

Na predição one-step, cada avaliação recebe o estado verdadeiro. No **rollout livre**, depois do estado inicial, usamos as **próprias predições**:
$$
\hat x_{k+1}=\hat F(\hat x_k,u_k).
$$

Utilizaremos a entrada $u_k=0.8\sin(0.2k)$, presente no experimento original de Koopman, durante 500 passos. Testaremos $x_0=[-0.6,0.6]^\top$, dentro da região de treino, e $x_0=[-0.6,1.4]^\top$, condição inicial histórica que começa **parcialmente fora**.

Propagamos **somente a média GP**: não afirmamos que os desvios-padrão de um passo sejam bandas probabilísticas válidas para toda a trajetória.


In [ ]:
N_ROLLOUT=500
U_roll=.8*np.sin(.2*np.arange(N_ROLLOUT))

def rollout(x0,U):
    real=np.empty((len(U)+1,2))
    predicted=np.empty_like(real)
    real[0]=np.asarray(x0,float)
    predicted[0]=np.asarray(x0,float)
    for k,u in enumerate(U):
        real[k+1]=simulate_step(real[k],u)
        predicted[k+1]=gp_step([[predicted[k,0],predicted[k,1],u]])[0][0]
    return real,predicted

def plot_rollout(x0,title):
    real,pred=rollout(x0,U_roll)
    t=DT*np.arange(len(real))
    fig,axes=plt.subplots(1,3,figsize=(14,3.7))
    for j,name in enumerate(["q","p"]):
        axes[j].plot(t,real[:,j],label="planta")
        axes[j].plot(t,pred[:,j],"--",label="GP")
        axes[j].set(xlabel="Tempo [s]",ylabel=name,title=title+" — "+name)
        axes[j].legend();axes[j].grid(alpha=.2)
    axes[2].plot(real[:,0],real[:,1],label="planta")
    axes[2].plot(pred[:,0],pred[:,1],"--",label="GP")
    axes[2].scatter([real[0,0]],[real[0,1]],marker="x",c="black")
    axes[2].set(xlabel="q",ylabel="p",title="Retrato de fase")
    axes[2].legend();axes[2].grid(alpha=.2)
    plt.tight_layout();plt.show()
    print(title,"RMSE do rollout [q,p]:",rmse(pred,real))
    return real,pred

real_in,pred_in=plot_rollout([-.6,.6],"Inicial dentro")
real_out,pred_out=plot_rollout([-.6,1.4],"Inicial fora")


## 7. Conclusão e limitações

**O que investigamos:** identificação discreta direta com dois GPs escalares, dados de estados completos, um passo, generalização, incerteza pontual e rollout livre.

**O que devemos distinguir:** erro local versus erro acumulado; GP versus baseline de persistência; regiões amostradas versus extrapolação; desvio-padrão do modelo versus erro real. A incerteza **não** garante um limite de erro e não foi propagada corretamente por múltiplos passos.

**Limitações:** (1) dois GPs independentes ignoram covariância cruzada entre saídas; (2) o termo diagonal é uma escolha de regularização apesar de não haver ruído de medição; (3) os hiperparâmetros são estimados pontualmente; (4) 400 amostras não constituem comparação justa com as 40 mil do experimento de Koopman; (5) a condição inicial de teste fora do domínio pode revelar viés de extrapolação, e não necessariamente deficiência do condicionamento gaussiano.

**Para explorar depois:** modificar o tamanho de treinamento, adicionar ruído de medição e comparar erros sob uma base experimental unificada com Koopman/SINDy. Só depois considerar estados não observados.

**Pergunta para nossa reunião:** em que circunstâncias um modelo com pequena predição de um passo, mas erro acumulado crescente, é útil para análise ou síntese de controle?
